From this python code ( in my own opinion i think 0.7 is optimal thereshold )

In [ ]:
import easyocr
from PIL import Image
import numpy as np
import os

# Initialize EasyOCR
reader = easyocr.Reader(['th'], gpu=False)
ALLOWLIST = "".join(chr(c) for c in range(0x0E01, 0x0E3B)) + "0123456789"

def run_easyocr_with_params(img, **params):
    """Run EasyOCR with custom parameters"""
    arr = np.array(img)
    
    # Default parameters
    default_params = {
        'detail': 1,
        'paragraph': False,
        'contrast_ths': 0.05,
        'adjust_contrast': 0.7,
        'text_threshold': 0.7,
        'low_text': 0.3,
        'link_threshold': 0.4,
        'allowlist': ALLOWLIST,
        'decoder': "beamsearch",
        'rotation_info': [0, 90, -90]
    }
    
    # Update with custom parameters
    default_params.update(params)
    
    results = reader.readtext(arr, **default_params)
    return results

def format_results(results):
    """Format OCR results for display"""
    if not results:
        return "ไม่พบข้อความ"
    
    texts = []
    for bbox, text, conf in results:
        texts.append(f"{text} ({conf:.3f})")
    
    return " | ".join(texts)

def test_parameter_combinations(image_path):
    """Test different parameter combinations"""
    
    print(f"=== ทดสอบ parameters กับ {os.path.basename(image_path)} ===\n")
    
    img = Image.open(image_path).convert("RGB")
    print(f"ขนาดรูป: {img.size}")
    
    # Test cases based on documentation recommendations
    test_cases = [
        {
            'name': 'Default (Lab settings)',
            'params': {
                'contrast_ths': 0.05,
                'adjust_contrast': 0.7,
                'text_threshold': 0.7,
                'low_text': 0.3,
                'link_threshold': 0.4
            }
        },
        {
            'name': 'High sensitivity (lower thresholds)',
            'params': {
                'contrast_ths': 0.01,  # More sensitive to contrast
                'adjust_contrast': 0.7,
                'text_threshold': 0.5,  # Lower text threshold
                'low_text': 0.2,       # Detect faint text
                'link_threshold': 0.3
            }
        },
        {
            'name': 'High contrast boost',
            'params': {
                'contrast_ths': 0.1,
                'adjust_contrast': 1.2,  # Increase contrast
                'text_threshold': 0.7,
                'low_text': 0.3,
                'link_threshold': 0.4
            }
        },
        {
            'name': 'Conservative (high thresholds)',
            'params': {
                'contrast_ths': 0.05,
                'adjust_contrast': 0.7,
                'text_threshold': 0.8,  # Higher confidence required
                'low_text': 0.4,
                'link_threshold': 0.5
            }
        },
        {
            'name': 'Small text optimized',
            'params': {
                'contrast_ths': 0.02,  # Very sensitive
                'adjust_contrast': 1.0,
                'text_threshold': 0.6,
                'low_text': 0.25,      # Detect small/faint text
                'link_threshold': 0.35
            }
        },
        {
            'name': 'Blurry image optimized',
            'params': {
                'contrast_ths': 0.08,
                'adjust_contrast': 1.5,  # High contrast boost
                'text_threshold': 0.5,   # Lower threshold for blurry text
                'low_text': 0.2,
                'link_threshold': 0.3
            }
        }
    ]
    
    results_summary = []
    
    for i, test_case in enumerate(test_cases, 1):
        print(f"{i}. {test_case['name']}")
        print(f"   Parameters: {test_case['params']}")
        
        try:
            results = run_easyocr_with_params(img, **test_case['params'])
            formatted_result = format_results(results)
            print(f"   Result: {formatted_result}")
            
            results_summary.append({
                'name': test_case['name'],
                'params': test_case['params'],
                'result': formatted_result,
                'count': len(results),
                'avg_confidence': np.mean([conf for _, _, conf in results]) if results else 0
            })
            
        except Exception as e:
            print(f"   Error: {e}")
            results_summary.append({
                'name': test_case['name'],
                'params': test_case['params'],
                'result': f"Error: {e}",
                'count': 0,
                'avg_confidence': 0
            })
        
        print()
    
    return results_summary

def compare_results(results_summary):
    """Compare and analyze results"""
    print("=== การเปรียบเทียบผลลัพธ์ ===")
    
    # Sort by average confidence
    sorted_results = sorted(results_summary, key=lambda x: x['avg_confidence'], reverse=True)
    
    print("\nเรียงตามความเชื่อมั่นเฉลี่ย:")
    for i, result in enumerate(sorted_results, 1):
        print(f"{i}. {result['name']}")
        print(f"   ความเชื่อมั่นเฉลี่ย: {result['avg_confidence']:.3f}")
        print(f"   จำนวนข้อความที่พบ: {result['count']}")
        print(f"   ผลลัพธ์: {result['result']}")
        print()
    
    print("=== คำแนะนำ ===")
    
    best_result = sorted_results[0]
    print(f"ผลลัพธ์ที่ดีที่สุด: {best_result['name']}")
    print(f"Parameters ที่แนะนำ:")
    for param, value in best_result['params'].items():
        print(f"  {param}: {value}")
    
    return best_result

def analyze_parameter_effects():
    """Analyze the effects of different parameters"""
    print("\n=== การวิเคราะห์ผลกระทบของ Parameters ===")
    
    explanations = {
        'contrast_ths': {
            'lower': 'ค่าต่ำ (0.01-0.02) = ไวต่อการเปลี่ยนแปลง contrast มากขึ้น, เหมาะกับรูปที่มี contrast ต่ำ',
            'higher': 'ค่าสูง (0.08-0.1) = ไวต่อการเปลี่ยนแปลง contrast น้อยลง, เหมาะกับรูปที่มี contrast ดีอยู่แล้ว'
        },
        'adjust_contrast': {
            'lower': 'ค่าต่ำ (<1.0) = ลด contrast, อาจเหมาะกับรูปที่ contrast สูงเกินไป',
            'higher': 'ค่าสูง (>1.0) = เพิ่ม contrast, เหมาะกับรูปที่เบลอหรือ contrast ต่ำ'
        },
        'text_threshold': {
            'lower': 'ค่าต่ำ (0.5-0.6) = detect ข้อความที่ไม่ชัดได้มากขึ้น แต่อาจได้ noise',
            'higher': 'ค่าสูง (0.7-0.8) = detect เฉพาะข้อความที่ชัดเจน, ลด noise แต่อาจพลาดข้อความบางส่วน'
        },
        'low_text': {
            'lower': 'ค่าต่ำ (0.2-0.25) = detect ข้อความเบาๆ หรือเล็กๆ ได้',
            'higher': 'ค่าสูง (0.3-0.4) = มาตรฐานสูงกว่า, detect เฉพาะข้อความที่ชัดเจน'
        }
    }
    
    for param, effects in explanations.items():
        print(f"\n{param}:")
        print(f"  • {effects['lower']}")
        print(f"  • {effects['higher']}")

# Example usage functions
def test_single_image(image_path):
    """Test a single image with different parameters"""
    results = test_parameter_combinations(image_path)
    best_params = compare_results(results)
    analyze_parameter_effects()
    return best_params

def test_multiple_images(image_folder):
    """Test multiple images and find best overall parameters"""
    image_files = [f for f in os.listdir(image_folder) 
                   if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    
    all_results = {}
    
    for img_file in image_files:
        print(f"\n{'='*60}")
        img_path = os.path.join(image_folder, img_file)
        results = test_parameter_combinations(img_path)
        all_results[img_file] = results
    
    # Analyze overall best parameters
    print(f"\n{'='*60}")
    print("=== สรุปผลรวมจากทุกรูป ===")
    
    # Count which parameter set performed best most often
    best_counts = {}
    for img_file, results in all_results.items():
        best = max(results, key=lambda x: x['avg_confidence'])
        best_name = best['name']
        best_counts[best_name] = best_counts.get(best_name, 0) + 1
    
    print("\nการนับผลลัพธ์ที่ดีที่สุดในแต่ละรูป:")
    for name, count in sorted(best_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"{name}: {count} ครั้ง")
    
    return all_results

# Main execution
if __name__ == "__main__":
    all_results = test_multiple_images("images")


Using CPU. Note: This module is much faster with a GPU.



=== ทดสอบ parameters กับ image01.jpg ===

ขนาดรูป: (199, 82)
1. Default (Lab settings)
   Parameters: {'contrast_ths': 0.05, 'adjust_contrast': 0.7, 'text_threshold': 0.7, 'low_text': 0.3, 'link_threshold': 0.4}
   Result: 1ก58107 (0.818) | กรงทพมหานคร (0.584)

2. High sensitivity (lower thresholds)
   Parameters: {'contrast_ths': 0.01, 'adjust_contrast': 0.7, 'text_threshold': 0.5, 'low_text': 0.2, 'link_threshold': 0.3}
   Result: 1ก58107 (0.868) | กรงทพมทานคร (0.553)

3. High contrast boost
   Parameters: {'contrast_ths': 0.1, 'adjust_contrast': 1.2, 'text_threshold': 0.7, 'low_text': 0.3, 'link_threshold': 0.4}
   Result: 1ก58107 (0.818) | กรงทพมหานคร (0.584)

4. Conservative (high thresholds)
   Parameters: {'contrast_ths': 0.05, 'adjust_contrast': 0.7, 'text_threshold': 0.8, 'low_text': 0.4, 'link_threshold': 0.5}
   Result: 1ก58107 (0.845) | กรงทพมหานคร (0.684)

5. Small text optimized
   Parameters: {'contrast_ths': 0.02, 'adjust_contrast': 1.0, 'text_threshold': 0.6, 'low_tex